### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\vijay\AppData\Local\Temp\ipykernel_21728\3494988372.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all pdfs in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

        except Exception as e:
            raise ValueError(e)

    print(f"\nTotal Documents Loaded: {len(all_documents)}")
    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Found 3 PDF files to process

Processing: Express.pdf

Processing: Node.pdf

Processing: React.pdf

Total Documents Loaded: 15


### Text splitting get into chunks

In [5]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print("\nExample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [10]:
chunks = split_documents(all_pdf_documents)
chunks

Split 15 documents into 32 chunks

Example Chunk:
Content: RAG Practice Dataset | Educational sample
Page 1
 Express.js: Routing, Middleware, and
 APIs
 A practical introduction to building HTTP services
Purpose: This document is a fictional, self-contained l...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-13T08:12:04+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-10-13T08:12:04+00:00', 'subject': 'Sample technical document for RAG and PDF loader practice', 'title': 'Express.js: Routing, Middleware, and APIs', 'trapped': '/False', 'source': '..\\data\\pdf\\Express.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Express.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-13T08:12:04+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-10-13T08:12:04+00:00', 'subject': 'Sample technical document for RAG and PDF loader practice', 'title': 'Express.js: Routing, Middleware, and APIs', 'trapped': '/False', 'source': '..\\data\\pdf\\Express.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Express.pdf', 'file_type': 'pdf'}, page_content='RAG Practice Dataset | Educational sample\nPage 1\n Express.js: Routing, Middleware, and\n APIs\n A practical introduction to building HTTP services\nPurpose: This document is a fictional, self-contained learning resource created for experimenting with PDF text\nextraction, chunking, embeddings, retrieval, and question answering in a Retrieval-Augmented Generation (RAG)\npipeline.\nContents\n1. What is Express.js?\n2. Routing and HTTP methods\n3. Middleware\n4. Validation an

### Embedding And vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Tuple, Any, Dict
from sklearn.metrics.pairwise import cosine_similarity


d:\PythonProjects\traditional_rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded sucessfully: {self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager = EmbeddingManager()


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17185.67it/s]


Model loaded sucessfully: 384


C:\Users\vijay\AppData\Local\Temp\ipykernel_21728\3057987722.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Model loaded sucessfully: {self.model.get_sentence_embedding_dimension()}"


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store",
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"},
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore = VectorStore()


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 32


In [9]:
### convert text to embeddings
texts = [doc.page_content for doc in chunks]


### Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

### Store in the vectorDB
vectorstore.add_documents(chunks, embeddings)

NameError: name 'chunks' is not defined

### Retriever Pipeline From VectorStore

In [10]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retrieve(self,query:str,top_k:int=5,score_threshold:float=0.0)->List[Dict[str,Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding=self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results=self.vector_store.collection.query(
               query_embeddings=query_embedding,
               n_results=top_k 
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i,(doc_id,document, metadata, distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score=1-distance

                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                print("No documents found")

            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)


In [ ]:
rag_retriever.retrieve("What is middleware")

### Integration VectorDB Context Pipeline with LLM Output

In [11]:
### Simple RAG Pipeline with Gemini LLM
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
load_dotenv()

# Initialize the LLM
llm = init_chat_model(
    "gemini-3.5-flash-lite",
    model_provider="google_genai",
    api_key=os.getenv("GEMINI_API_KEY"),
    max_tokens=1024,
    temperature=0.1
)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## Retrieve the context
    results=retriever.retrieve(query=query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else "" 
    if not context:
        return "No relevant context found to answer the query"

    #Generate the answer using the LLM
    prompt=f"""Use the following context to answer the query concisely.

        Context:
            {context}

        Question:{query}

        Answer:"""

    response = llm.invoke(prompt)
    return response.content[0]["text"]

In [25]:
answer=rag_simple("What is nodejs",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is nodejs'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.19it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)



d:\PythonProjects\traditional_rag\.venv\lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Node.js is a JavaScript runtime built on the V8 engine that allows JavaScript to run outside a browser. It uses an event-driven architecture and non-blocking I/O, and is commonly used for APIs, command-line tools, real-time services, and background workers.


### Enhanced RAG Pipeline

In [16]:
def rag_advanced(query,retriever,llm,top_k=5,min_score=0.2,return_context=False):
    """
        RAG Pipeline with extra features
        - Returns answer, sources, confidence score, and optionally full context
    """

    results=retriever.retrieve(query,top_k=top_k,score_threshold=min_score)

    if not results:
        return {'answer':"No relevant context found",'sources':[],'confidence':0.0,'context':''}

    #Prepare context and sources
    context="\n\n".join([doc['content'] for doc in results])
    sources=[{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke(prompt)

    output = {
        'answer': response.content[0]["text"],
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output
    

In [ ]:
#Example Usage
result = rag_advanced("What is nodejs", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")
print(f"Confidence: {result['confidence']}")
print("Context Preview:", result['context'][:300])

In [14]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import Dict,List,Any
import time

class AdvancedRAGPipeline:

    def __init__(self,retriever,llm):
        self.retriever=retriever
        self.llm=llm
        self.history = []  # Store query history

    def query(self,question:str,top_k:int=5,min_score: float = 0.2,stream:bool=False,summarize: bool = False)->Dict[str,Any]:
        # Retrieve relevant documents
        results=self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context="\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""

            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke(prompt)
            answer = response.content[0]["text"]

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content[0]["text"]

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }



In [15]:
# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is react", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is react'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.34it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
RAG Practice Dataset | Educational sample
Page 2
1. What is React?
React is a JavaScript li

brary for building user interfaces from reusable components. A component describes a part
of the interface and returns a tree of elements that React can render. React is commonly used for single-page
applications, dashboards, forms, and interactive product interfaces.
React focuses on the view layer. Routing, server communication, global state, and build tooling are typically
handled by additional libraries or platform features. A React application can be created with a modern build tool or
integrated into an existing application.
A minimal function component
 function Welcome({ name }) {
  return &lt;h1&gt;Hello, {name}!&lt;/h1&gt;;
}
export default Welcome;
2. Components and props
Components are reusable units of UI. Component names conventionally begin with a capital letter. Props are
read-only inputs passed from a parent component to a child component. They help a component render different

Question: what is react

Answer:


d:\PythonProjects\traditional_rag\.venv\lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\PythonProjects\traditional_rag\.venv\lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Final Answer: React is a JavaScript library for building user interfaces from reusable components.

Citations:
[1] React.pdf (page 1)
Summary: React is a JavaScript library used to create user interfaces. It achieves this by utilizing reusable components.
History: {'question': 'what is react', 'answer': 'React is a JavaScript library for building user interfaces from reusable components.', 'sources': [{'source': 'React.pdf', 'page': 1, 'score': 0.24695688486099243, 'preview': 'RAG Practice Dataset | Educational sample\nPage 2\n1. What is React?\nReact is a JavaScript library for building user inter...'}], 'summary': 'React is a JavaScript library used to create user interfaces. It achieves this by utilizing reusable components.'}
